# Phase 4 — Modeling

**Goal:** Train a LightGBM fraud classifier on the engineered features from Phase 3.

**Modeling decisions (locked from EDA):**
- Algorithm: LightGBM. Reasons: handles NaN natively, fast on 472K rows, captures feature interactions tree-based models excel at.
- Imbalance handling: `scale_pos_weight=28` (≈ neg/pos ratio). No resampling — for tree models on severe imbalance, class weights typically match or outperform SMOTE while training faster.
- Evaluation metric for early stopping: **PR-AUC** (average_precision). Not ROC-AUC, because the high class imbalance makes ROC-AUC look misleadingly high — a useless model can still achieve ROC-AUC 0.90 on 3.5%-positive data.
- Operational metric: **Recall @ 1% FPR**. Translates to "if the ops team can review 1% of legit transactions, what fraction of frauds do we catch?"
- Threshold: optimized in Session 2 against a business cost matrix ($100 per missed fraud, $5 per false alarm).

In [1]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow

# Add project root to path
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.models.train import train_lightgbm, compute_metrics, recall_at_fpr

# Paths
PROCESSED = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"
FIGURES = ROOT / "reports" / "figures"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# MLflow tracking URI — store experiments in project root
mlflow.set_tracking_uri(f"file:{ROOT / 'mlruns'}")

# Plot styling
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (10, 6)

print("Setup complete.")
print(f"MLflow tracking: {mlflow.get_tracking_uri()}")

Setup complete.
MLflow tracking: file:C:\Projects\fraud-detection-system\mlruns


In [2]:
# Load engineered features
print("Loading engineered features from Phase 3 artifacts...")
X_train = pd.read_parquet(PROCESSED / "X_train.parquet")
X_val = pd.read_parquet(PROCESSED / "X_val.parquet")
y_train = pd.read_parquet(PROCESSED / "y_train.parquet")["isFraud"]
y_val = pd.read_parquet(PROCESSED / "y_val.parquet")["isFraud"]

print(f"X_train: {X_train.shape}  |  fraud rate: {y_train.mean()*100:.2f}%")
print(f"X_val:   {X_val.shape}  |  fraud rate: {y_val.mean()*100:.2f}%")

# Reapply categorical dtypes from fitted pipeline (Parquet round-trip prunes them)
import pickle
with open(MODELS_DIR / "feature_engineer.pkl", "rb") as f:
    fe = pickle.load(f)
for col, cats in fe.cat_categories_.items():
    if col in X_train.columns:
        X_train[col] = pd.Categorical(X_train[col].astype("object"), categories=cats)
    if col in X_val.columns:
        X_val[col] = pd.Categorical(X_val[col].astype("object"), categories=cats)

print("\nCategorical columns after restoration:")
for col in X_train.select_dtypes(include="category").columns:
    print(f"  {col:<15} categories={list(X_train[col].cat.categories)}")

Loading engineered features from Phase 3 artifacts...
X_train: (472432, 36)  |  fraud rate: 3.51%
X_val:   (118108, 36)  |  fraud rate: 3.44%

Categorical columns after restoration:
  ProductCD       categories=['C', 'H', 'R', 'S', 'W']
  card4           categories=['american express', 'discover', 'mastercard', 'missing', 'visa']
  card6           categories=['charge card', 'credit', 'debit', 'debit or credit', 'missing']
  DeviceType      categories=['desktop', 'missing', 'mobile']


In [3]:
# Baseline 1: Naive "always predict not fraud" — what's the accuracy?
naive_accuracy = (y_val == 0).mean()
print(f"Baseline 'always predict not fraud':")
print(f"  Accuracy: {naive_accuracy*100:.2f}%")
print(f"  Recall:   0.00% (catches no fraud)")
print(f"  This is why we don't use accuracy.\n")

# Baseline 2: Random scores (uniform)
np.random.seed(42)
random_scores = np.random.rand(len(y_val))
baseline_metrics = compute_metrics(y_val.values, random_scores, threshold=0.5)
print(f"Baseline 'random scores':")
print(f"  PR-AUC:           {baseline_metrics['pr_auc']:.4f}  (≈ fraud rate {y_val.mean():.4f})")
print(f"  ROC-AUC:          {baseline_metrics['roc_auc']:.4f}  (random = 0.5)")
print(f"  Recall @ 1% FPR:  {baseline_metrics['recall_at_1pct_fpr']:.4f}")

Baseline 'always predict not fraud':
  Accuracy: 96.56%
  Recall:   0.00% (catches no fraud)
  This is why we don't use accuracy.

Baseline 'random scores':
  PR-AUC:           0.0343  (≈ fraud rate 0.0344)
  ROC-AUC:          0.4983  (random = 0.5)
  Recall @ 1% FPR:  0.0091


In [4]:
# First real model — default hyperparameters
t0 = time.time()
model, metrics = train_lightgbm(
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    run_name="lgbm-baseline-default-params",
)
elapsed = time.time() - t0

print(f"\n=== Training complete in {elapsed:.1f}s ===\n")
print(f"Best iteration: {model.best_iteration}")
print(f"\nValidation metrics:")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k:<25} {v:.4f}")
    else:
        print(f"  {k:<25} {v}")

[100]	train's average_precision: 0.599667	val's average_precision: 0.437105
[200]	train's average_precision: 0.661444	val's average_precision: 0.452013
[300]	train's average_precision: 0.710931	val's average_precision: 0.455018
[400]	train's average_precision: 0.751027	val's average_precision: 0.457041
[500]	train's average_precision: 0.784438	val's average_precision: 0.458744
[600]	train's average_precision: 0.812032	val's average_precision: 0.461471
[700]	train's average_precision: 0.835815	val's average_precision: 0.463461
[800]	train's average_precision: 0.858068	val's average_precision: 0.465607
[900]	train's average_precision: 0.877213	val's average_precision: 0.466871
[1000]	train's average_precision: 0.893977	val's average_precision: 0.468708
[1100]	train's average_precision: 0.909009	val's average_precision: 0.469909
[1200]	train's average_precision: 0.923254	val's average_precision: 0.471018
[1300]	train's average_precision: 0.933703	val's average_precision: 0.472287


2026/05/21 04:00:32 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



=== Training complete in 74.9s ===

Best iteration: 1250

Validation metrics:
  pr_auc                    0.4727
  roc_auc                   0.8983
  precision                 0.3382
  recall                    0.5984
  f1                        0.4321
  n_predicted_positive      7192
  threshold                 0.5000
  recall_at_1pct_fpr        0.4134


In [5]:
# Check that MLflow logged this run properly
from mlflow.tracking import MlflowClient

client = MlflowClient()
experiment = client.get_experiment_by_name("fraud-detection")
runs = client.search_runs(experiment_ids=[experiment.experiment_id], order_by=["start_time DESC"])

print(f"Found {len(runs)} run(s) in 'fraud-detection' experiment\n")
for i, run in enumerate(runs[:3]):
    print(f"Run {i+1}: {run.data.tags.get('mlflow.runName', 'unnamed')}")
    print(f"  Status: {run.info.status}")
    print(f"  PR-AUC: {run.data.metrics.get('pr_auc', 'N/A'):.4f}")
    print(f"  Recall@1%FPR: {run.data.metrics.get('recall_at_1pct_fpr', 'N/A'):.4f}")
    print()

Found 3 run(s) in 'fraud-detection' experiment

Run 1: lgbm-baseline-default-params
  Status: FINISHED
  PR-AUC: 0.4727
  Recall@1%FPR: 0.4134

Run 2: lgbm-baseline-default-params
  Status: FAILED
  PR-AUC: 0.4727
  Recall@1%FPR: 0.4134

Run 3: lgbm-baseline-default-params
  Status: FAILED
  PR-AUC: 0.4727
  Recall@1%FPR: 0.4134

